# Dataframe Operations

In [1]:
import sys
print(sys.executable)

/home/vishwajeet/spark/.venv/bin/python3


In [2]:
from pyspark.sql import SparkSession

In [3]:
from pyspark.sql.functions import desc, col

In [4]:
# spark = SparkSession.builder\
#     # .config("spark.pyspark.python", sys.executable) \
#     # .config("spark.pyspark.driver.python", sys.executable) \
#     .appName("DataframesOp")\
#     .getOrCreate()

In [5]:
spark = SparkSession.builder.appName("DataframesOp").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 01:19:57 WARN Utils: Your hostname, LAPTOP-EC11ARA9, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/02 01:19:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 01:19:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
file_path = "./data/stock_data.txt"

In [7]:
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [8]:
df.printSchema()
print("Initial Dataframe:")
df.show(10)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)

Initial Dataframe:
+---+-------------+-----------+--------+-----+
| id|         name|   category|quantity|price|
+---+-------------+-----------+--------+-----+
|  1|   Monitor 34|  Furniture|     353| 1637|
|  2|   Monitor 83|  Furniture|     498| 1983|
|  3|      Desk 37|  Furniture|      62| 1779|
|  4|    Tablet 96|  Furniture|     453|  574|
|  5|Headphones 53|Electronics|     160|  721|
|  6|Headphones 29|Electronics|     138| 1222|
|  7|     Mouse 91|  Furniture|     177|  996|
|  8|   Monitor 28|Electronics|     312|  741|
|  9|     Phone 38|  Furniture|     110|  188|
| 10|   Monitor 66|Accessories|     147|  910|
+---+-------------+-----------+--------+-----+
only showing top 10 rows


## Select Columns

In [9]:
selected_df = df.select("id", "name", "price")

In [10]:
print(selected_df)
selected_df.show(10)

DataFrame[id: int, name: string, price: int]
+---+-------------+-----+
| id|         name|price|
+---+-------------+-----+
|  1|   Monitor 34| 1637|
|  2|   Monitor 83| 1983|
|  3|      Desk 37| 1779|
|  4|    Tablet 96|  574|
|  5|Headphones 53|  721|
|  6|Headphones 29| 1222|
|  7|     Mouse 91|  996|
|  8|   Monitor 28|  741|
|  9|     Phone 38|  188|
| 10|   Monitor 66|  910|
+---+-------------+-----+
only showing top 10 rows


## Filtering Rows

In [11]:
filtered_df = df.filter(100<df.quantity)

In [12]:
filtered_df.show(10)

+---+-------------+-----------+--------+-----+
| id|         name|   category|quantity|price|
+---+-------------+-----------+--------+-----+
|  1|   Monitor 34|  Furniture|     353| 1637|
|  2|   Monitor 83|  Furniture|     498| 1983|
|  4|    Tablet 96|  Furniture|     453|  574|
|  5|Headphones 53|Electronics|     160|  721|
|  6|Headphones 29|Electronics|     138| 1222|
|  7|     Mouse 91|  Furniture|     177|  996|
|  8|   Monitor 28|Electronics|     312|  741|
|  9|     Phone 38|  Furniture|     110|  188|
| 10|   Monitor 66|Accessories|     147|  910|
| 11|     Mouse 93|Electronics|     104| 1846|
+---+-------------+-----------+--------+-----+
only showing top 10 rows


### Advance Filtering

In [13]:
adv_filterd_df = df.filter((100<df.quantity) & (df.quantity<200)) 

In [14]:
adv_filterd_df.show(10)

+---+-------------+-----------+--------+-----+
| id|         name|   category|quantity|price|
+---+-------------+-----------+--------+-----+
|  5|Headphones 53|Electronics|     160|  721|
|  6|Headphones 29|Electronics|     138| 1222|
|  7|     Mouse 91|  Furniture|     177|  996|
|  9|     Phone 38|  Furniture|     110|  188|
| 10|   Monitor 66|Accessories|     147|  910|
| 11|     Mouse 93|Electronics|     104| 1846|
| 16|      Chair 7|  Furniture|     127| 1651|
| 32|   Printer 77|Accessories|     113| 1385|
| 35|Headphones 12|Accessories|     194| 1072|
| 40|    Tablet 98|Electronics|     182|  287|
+---+-------------+-----------+--------+-----+
only showing top 10 rows


## Aggregation on Grouped Data

In [15]:
grouped_data = df.groupBy("category").agg({"quantity":"sum", "price":"avg"})

In [16]:
grouped_data.show()

+-----------+-------------+------------------+
|   category|sum(quantity)|        avg(price)|
+-----------+-------------+------------------+
|Electronics|        77849|1011.2100313479624|
|  Furniture|        92046|1022.7303370786517|
|Accessories|        85391|           1028.36|
+-----------+-------------+------------------+



## Joining Dataframes based on specified columns

In [17]:
df1 = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
], ["id", "name"])

df2 = spark.createDataFrame([
    (1, "Pune"),
    (2, "Mumbai"),
    (4, "Delhi")
], ["id", "city"])

In [18]:
print("df1:")
df1.show()
print("df2:")
df2.show()


df1:


+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+

df2:
+---+------+
| id|  city|
+---+------+
|  1|  Pune|
|  2|Mumbai|
|  4| Delhi|
+---+------+



In [19]:
joined_df = df2.join(df1, "id", "inner")

In [20]:
joined_df.show()

+---+------+-----+
| id|  city| name|
+---+------+-----+
|  1|  Pune|Alice|
|  2|Mumbai|  Bob|
+---+------+-----+



In [21]:
joined_df2 = df2.join(df1, "id", "left") #df2 is on left

In [22]:
joined_df2.show()

+---+------+-----+
| id|  city| name|
+---+------+-----+
|  1|  Pune|Alice|
|  2|Mumbai|  Bob|
|  4| Delhi| NULL|
+---+------+-----+



## Sort using OrderBy

In [23]:
sort = df.orderBy("price", ascending=False) #default True

In [24]:
sort.show(10)

+---+-------------+-----------+--------+-----+
| id|         name|   category|quantity|price|
+---+-------------+-----------+--------+-----+
|324|Headphones 58|Accessories|     273| 1996|
|813|Headphones 42|  Furniture|     377| 1996|
|676|     Phone 44|Electronics|     280| 1995|
|123|      Desk 46|Accessories|     273| 1992|
|196|  Keyboard 41|Electronics|      88| 1990|
|576|      Desk 41|Accessories|      58| 1990|
|428|   Printer 90|Electronics|     194| 1988|
|370|    Printer 6|Accessories|     141| 1987|
|639|      Desk 80|Accessories|     398| 1987|
|579|     Laptop 6|Accessories|     121| 1987|
+---+-------------+-----------+--------+-----+
only showing top 10 rows


### Sort by Column Advanced

In [25]:
sorted_df = df.orderBy(col("category").desc(), col("price").desc()) #first sorts by category then within by price

In [26]:
sorted_df.show(10)

+---+-------------+---------+--------+-----+
| id|         name| category|quantity|price|
+---+-------------+---------+--------+-----+
|813|Headphones 42|Furniture|     377| 1996|
|701|   Printer 40|Furniture|      71| 1987|
|999|     Phone 54|Furniture|      41| 1984|
|  2|   Monitor 83|Furniture|     498| 1983|
|450|     Mouse 33|Furniture|     131| 1975|
|552|     Chair 98|Furniture|     330| 1972|
|338|     Mouse 73|Furniture|     232| 1968|
|950|     Phone 73|Furniture|      84| 1966|
|351|Headphones 47|Furniture|     286| 1961|
|781|     Chair 73|Furniture|      94| 1961|
+---+-------------+---------+--------+-----+
only showing top 10 rows


## Distinct Row

In [27]:
distinct_rows = df.select("category").distinct()

In [28]:
distinct_rows.show()

+-----------+
|   category|
+-----------+
|Electronics|
|  Furniture|
|Accessories|
+-----------+



## Drop Columns

In [29]:
dropped_col = df.drop("category")

In [30]:
dropped_col.show()

+---+-------------+--------+-----+
| id|         name|quantity|price|
+---+-------------+--------+-----+
|  1|   Monitor 34|     353| 1637|
|  2|   Monitor 83|     498| 1983|
|  3|      Desk 37|      62| 1779|
|  4|    Tablet 96|     453|  574|
|  5|Headphones 53|     160|  721|
|  6|Headphones 29|     138| 1222|
|  7|     Mouse 91|     177|  996|
|  8|   Monitor 28|     312|  741|
|  9|     Phone 38|     110|  188|
| 10|   Monitor 66|     147|  910|
| 11|     Mouse 93|     104| 1846|
| 12|Headphones 45|     333|  246|
| 13|      Desk 78|     383| 1314|
| 14|     Mouse 58|     315| 1848|
| 15|   Printer 89|       9| 1347|
| 16|      Chair 7|     127| 1651|
| 17|  Keyboard 81|     290|  205|
| 18|    Tablet 57|      91|  314|
| 19|      Chair 5|     286| 1915|
| 20|Headphones 22|     454| 1727|
+---+-------------+--------+-----+
only showing top 20 rows


## Add New Calculated Column

In [31]:
added_col = df.withColumn("revenue", df.quantity * df.price)

In [32]:
added_col.show()

+---+-------------+-----------+--------+-----+-------+
| id|         name|   category|quantity|price|revenue|
+---+-------------+-----------+--------+-----+-------+
|  1|   Monitor 34|  Furniture|     353| 1637| 577861|
|  2|   Monitor 83|  Furniture|     498| 1983| 987534|
|  3|      Desk 37|  Furniture|      62| 1779| 110298|
|  4|    Tablet 96|  Furniture|     453|  574| 260022|
|  5|Headphones 53|Electronics|     160|  721| 115360|
|  6|Headphones 29|Electronics|     138| 1222| 168636|
|  7|     Mouse 91|  Furniture|     177|  996| 176292|
|  8|   Monitor 28|Electronics|     312|  741| 231192|
|  9|     Phone 38|  Furniture|     110|  188|  20680|
| 10|   Monitor 66|Accessories|     147|  910| 133770|
| 11|     Mouse 93|Electronics|     104| 1846| 191984|
| 12|Headphones 45|Electronics|     333|  246|  81918|
| 13|      Desk 78|  Furniture|     383| 1314| 503262|
| 14|     Mouse 58|  Furniture|     315| 1848| 582120|
| 15|   Printer 89|Electronics|       9| 1347|  12123|
| 16|     

## Rename Column

In [33]:
renamed_col = df.withColumnRenamed("price", "product_price")

In [34]:
renamed_col.show(10)

+---+-------------+-----------+--------+-------------+
| id|         name|   category|quantity|product_price|
+---+-------------+-----------+--------+-------------+
|  1|   Monitor 34|  Furniture|     353|         1637|
|  2|   Monitor 83|  Furniture|     498|         1983|
|  3|      Desk 37|  Furniture|      62|         1779|
|  4|    Tablet 96|  Furniture|     453|          574|
|  5|Headphones 53|Electronics|     160|          721|
|  6|Headphones 29|Electronics|     138|         1222|
|  7|     Mouse 91|  Furniture|     177|          996|
|  8|   Monitor 28|Electronics|     312|          741|
|  9|     Phone 38|  Furniture|     110|          188|
| 10|   Monitor 66|Accessories|     147|          910|
+---+-------------+-----------+--------+-------------+
only showing top 10 rows


In [35]:
spark.stop()

In [36]:
# import sys
# print(sys.executable)